# Hyperliquid consolidated account analytics

Combines balances, account history, fills, funding, positions, and PnL across the configured Hyperliquid wallets. Combined account-value history is shown only over the dates available for every wallet.

Daily rollups and calendar-period cutoffs use `America/New_York`. Previously saved UTC-aggregated daily totals are mapped to Eastern date labels when loaded; their original intraday split cannot be reconstructed.


## Wallets and implementation

Edit `WALLET_ADDRESSES` here. Both addresses found in the single-wallet notebook are included: the active address and the previously commented address.


In [11]:
import json
from pathlib import Path

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]

RISK_BUDGET_PCT = 5.0
ATR_PERIOD = 14
ATR_STOP_MULTIPLE = 1.25
POSITION_TAKE_PROFITS = {
    # ('BTC', 'LONG'): 110000,
}

namespace={'__name__':'hyperliquid_multi_account_analytics'}
implementation_paths=[
    Path('hyperliquid_account_analytics_implementation.ipynb'),
    Path('hyperliquid_multi_account_analytics_implementation.ipynb'),
]
for implementation_path in implementation_paths:
    implementation=json.loads(implementation_path.read_text(encoding='utf-8'))
    for cell in implementation['cells']:
        if cell['cell_type']=='code':
            source=''.join(cell['source'])
            if source.strip():
                exec(compile(source,f'{implementation_path} cell','exec'),namespace)
MultiWalletAnalytics=namespace['MultiWalletAnalytics']
analytics=MultiWalletAnalytics(WALLET_ADDRESSES,data_dir='data')
analytics.position_risk_budget_pct=RISK_BUDGET_PCT
analytics.position_take_profit_levels=POSITION_TAKE_PROFITS
analytics.position_atr_period=ATR_PERIOD
analytics.position_atr_stop_multiple=ATR_STOP_MULTIPLE


## Fetch and consolidate wallet data

Fetch each wallet once and prepare combined account history and fills.


In [12]:
_ = analytics.fetch_data()


hyperliquid_account_analytics_implementation.ipynb cell:72: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
hyperliquid_account_analytics_implementation.ipynb cell:75: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
hyperliquid_account_analytics_implementation.ipynb cell:72: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
hyperliquid_account_analytics_implementation.ipynb cell:75: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
hyperliquid_account_analytics_implementation.ipynb cell:72: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
hyperliquid_account_analytics_implementation.ipynb cell:75: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.


Loaded 2 wallets; 647 available fills; combined current account value $2,975.57.
Combined historical account value uses shared daily coverage in America/New_York from 2026-10-25 through 2026-10-28.


## Account / PnL

Combined account and risk summary. Daily PnL and calendar-period cutoffs use US Eastern time.


In [13]:
analytics.show_account_pnl()


=== ACCOUNT / PNL AND RISK ===


Metric,Value
Account value,"$2,918.78"
Portfolio value,"$2,975.57 (combined live wallet value)"
Account 1D (ET),2.39%
Realized today (ET),$27.87
Realized today (% of account value),0.95%
Unrealized PnL,$-180.68
Unrealized PnL (% of account value),-6.19%
Unrealized 1D change (ET),n/a
Average realized PnL / day,$-0.96
Average unrealized PnL change / day,n/a


## Account charts

Combined account value and PnL charts.


In [14]:
analytics.show_account_charts()


## Account vs BTC

Combined account comparison against BTC.


In [15]:
analytics.show_account_vs_btc()


Lifetime account series is anchored to current account value; it includes fee-net realized trades and perp funding, but excludes historical cash flows and unrealized PnL.


## Traded volume vs realized PnL chart

Daily fill volume and realized PnL pooled across wallets.


In [16]:
analytics.show_volume_chart()


## Open positions

Show each wallet position separately, followed by fixed-fractional sizing grouped by asset and side and an aggregate summary. Set `RISK_BUDGET_PCT` (5% per asset), `ATR_PERIOD` (14 completed daily candles), and `ATR_STOP_MULTIPLE` (1.25) in the setup cell. Stops are placed 1.25 ATR from the current mark. Same-side positions across wallets are combined; long and short rows remain separate. If both sides are open for one asset, they share that asset's risk budget equally. Optional take-profit prices can be entered in `POSITION_TAKE_PROFITS`, for example `('BTC', 'LONG'): 110000`.


In [17]:
analytics.show_open_positions()


Wallet,Coin,Side,Liq %,ROE,uPnL,Funding,Funding (1h),Funding APR,Entry,Mark,Pos,Margin,Margin Type,Margin %,Fees,Liq Px,Lev.,Sz
0xf3fb…c248,XPL,SHORT,14.28%,-0.28%,-$0.56,$0.87,0.007146%,62.60%,0.081621,0.081663,"$1,011",$202,Isolated,34.81%,-$0.34,0.093322264,5.00x,"-12,380.00"
0xf3fb…c248,IMX,SHORT,40.25%,3.03%,$6.00,$0.18,0.001250%,10.95%,0.12332,0.12258,$986,$197,Cross,33.96%,-$0.14,0.17192326,5.00x,"-8,040.50"
0x7532…07f2,xyz:JIO,LONG,7.22%,-9.54%,-$9.33,$24.50,0.006284%,55.05%,14.22973,14.094,$968,$115,Isolated,12.01%,-$0.07,13.076198,8.43x,68.70
0x7532…07f2,xyz:GOOGL,LONG,n/a,-52.72%,-$102.30,-$2.26,0.000625%,5.48%,343.02,282.74,$480,$160,Cross,16.72%,-$0.05,0,3.00x,1.70
0x7532…07f2,ADA,SHORT,206.01%,12.03%,$4.75,-$0.16,-0.003957%,-34.67%,0.195188,0.192839,$390,$39,Cross,4.08%,-$0.11,0.59010536,10.00x,"-2,022.00"
0x7532…07f2,io:PRL,SHORT,16.67%,4.26%,$4.60,$2.64,0.029501%,258.43%,1.48825,1.4671,$319,$115,Isolated,12.05%,-$0.03,1.7116075,2.77x,-217.60
0xf3fb…c248,XTEN,SHORT,11.20%,-8.91%,-$8.67,$0.23,0.001250%,10.95%,0.273429,0.28155,$301,$89,Isolated,15.40%,-$0.09,0.31309599,3.36x,"-1,068.00"
0x7532…07f2,BANANA,SHORT,255.19%,-0.88%,-$0.83,$0.11,0.001250%,10.95%,3.2282,3.23771,$283,$94,Cross,9.87%,-$0.05,11.499952,3.00x,-87.50
0xf3fb…c248,INIT,SHORT,14.86%,1.45%,$1.31,$-0.00,-0.008215%,-71.97%,0.18359,0.18265,$269,$91,Isolated,15.75%,-$0.04,0.20979448,2.94x,"-1,473.00"
0x7532…07f2,EIGEN,SHORT,361.92%,-14.00%,-$7.17,$0.08,0.001250%,10.95%,0.1848,0.1912,$212,$53,Cross,5.54%,-$0.05,0.88318445,4.00x,"-1,108.22"


Metric,Value
Wallets,2
Position value,"$6,635"
uPnL,$-180.68
ROE,-0.12%
Margin,"$1,537"
Margin % bal,0.49%
Leverage (Pos / Margin),4.32x
Funding,$26.80
Account value,"$2,975.57"


## Realized PnL statistics

Pool spot/perp fills and signed funding across wallets.


In [18]:
analytics.show_realized_pnl()


=== REALIZED SPOT ===


Metric,Value
Realized PnL,$22.35 (0.75%)
Funding,n/a
24H,$0.00 (0.00%)
7D,$0.00 (0.00%)
30D,$22.35 (0.75%)
QTD,$22.35 (0.75%)
YTD,$22.35 (0.75%)
W,3
Win %,100.00%
Avg W,$7.45


=== REALIZED PERP ===


Metric,Value
Realized PnL,$-60.53 (-2.03%)
Funding,$-56.23 (-1.89%)
24H,$27.87 (0.94%)
7D,$357.23 (12.01%)
30D,$463.43 (15.57%)
QTD,$463.39 (15.57%)
YTD,$463.27 (15.57%)
W,169
Win %,77.17%
Avg W,$9.19



=== PERP PNL BREAKDOWN ===


Component,Amount ($),Contribution (% of |net PnL|)
Trading,$100.21,165.56%
Fees,$-104.50,-172.65%
Funding,$-56.23,-92.90%
Net perp PnL,$-60.53,


Liquidations: 0 of 219 closes (0.00%) | liquidation PnL: $0.00.


## Traded volume vs PnL

Combined turnover efficiency and correlations.


In [19]:
analytics.show_volume_summary()


=== TRADED VOLUME VS REALIZED PNL (ALL WALLETS) ===


Market,Volume ($),Realized PnL ($),Realized PnL / Volume (%),PnL / $1k volume ($),Active days,Daily Pearson r,Daily Spearman r
Perps,"$243,825.24",$-4.30,-0.00%,$-0.02,56,0.101,0.290
Spot,"$1,397.53",$22.35,1.60%,$16.00,5,-0.509,-0.462


## PnL attribution

Combined realized and open-position PnL by asset.


In [20]:
analytics.show_pnl_attribution()


=== CONSOLIDATED PNL ATTRIBUTION ===


Type,Asset,Closes,Realized,Realized (% gross),uPnL,uPnL (% gross),Total PnL,Total (% gross)
Perp,BTC,11,$-474.42,-19.05%,$0.00,0.00%,$-474.42,-17.67%
Perp,kBONK,3,$-397.06,-15.95%,$0.00,0.00%,$-397.06,-14.79%
Perp,ZORA,21,$304.49,12.23%,$0.00,0.00%,$304.49,11.34%
Perp,KAITO,4,$-193.84,-7.78%,$0.00,0.00%,$-193.84,-7.22%
Perp,HYPE,11,$146.54,5.89%,$0.00,0.00%,$146.54,5.46%
Perp,ETH,14,$116.62,4.68%,$0.00,0.00%,$116.62,4.34%
Perp,VVV,40,$116.26,4.67%,$0.00,0.00%,$116.26,4.33%
Perp,xyz:GOOGL,0,$-2.59,-0.10%,$-102.30,-43.74%,$-104.89,-3.91%
Perp,io:ANTH,2,$-86.69,-3.48%,$0.00,0.00%,$-86.69,-3.23%
Perp,AAVE,4,$54.17,2.18%,$0.00,0.00%,$54.17,2.02%


Net attributed PnL across wallets: $-218.80.
